# Response Length Distribution Analysis

This notebook analyzes the distribution of response lengths for the Mac-a-Thon application questions.

In [ ]:
# Import Required Libraries
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy import stats

In [ ]:
# Connect to the database
conn = sqlite3.connect('../../db/applications.csv.db')

# Query 1: Length of "about yourself" response
query1 = '''
SELECT LENGTH("Answer this short answer question in 5 sentences or less: Tell us about yourself, and why you would like to attend the Mac-a-Thon.") AS "about_length"
FROM applications
ORDER BY "about_length" ASC;
'''

# Query 2: Length of "project" response
query2 = '''
SELECT LENGTH("Answer this short answer question in 5 sentences or less: What is a project that you recently worked on? This does not have to be related to computer science or software.") AS "project_length"
FROM applications
ORDER BY "project_length" ASC;
'''

# Query 3: Combined length of both responses
query3 = '''
SELECT (LENGTH("Answer this short answer question in 5 sentences or less: Tell us about yourself, and why you would like to attend the Mac-a-Thon.") +
    LENGTH("Answer this short answer question in 5 sentences or less: What is a project that you recently worked on? This does not have to be related to computer science or software.")) AS "response_length"
FROM applications
ORDER BY "response_length" ASC;
'''

# Load data
df_about = pd.read_sql_query(query1, conn)
df_project = pd.read_sql_query(query2, conn)
df_combined = pd.read_sql_query(query3, conn)

conn.close()

print(f"About Response - Mean: {df_about['about_length'].mean():.2f}, Std: {df_about['about_length'].std():.2f}")
print(f"Project Response - Mean: {df_project['project_length'].mean():.2f}, Std: {df_project['project_length'].std():.2f}")
print(f"Combined Response - Mean: {df_combined['response_length'].mean():.2f}, Std: {df_combined['response_length'].std():.2f}")

In [ ]:
# Create distribution visualizations with fitted normal curves
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        'Response Length Distribution: "About Yourself" Question',
        'Response Length Distribution: "Recent Project" Question',
        'Response Length Distribution: Combined Responses'
    ),
    vertical_spacing=0.12
)

# Function to add histogram with normal curve
def add_distribution(fig, data, column_name, row, color):
    # Add histogram
    fig.add_trace(
        go.Histogram(
            x=data[column_name],
            name='Actual',
            histnorm='probability density',
            marker_color=color,
            opacity=0.7,
            showlegend=(row == 1)
        ),
        row=row, col=1
    )
    
    # Fit normal distribution
    mu = data[column_name].mean()
    sigma = data[column_name].std()
    x_range = np.linspace(data[column_name].min(), data[column_name].max(), 100)
    normal_curve = stats.norm.pdf(x_range, mu, sigma)
    
    # Add normal curve
    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=normal_curve,
            name='Normal Fit',
            line=dict(color='red', width=2),
            showlegend=(row == 1)
        ),
        row=row, col=1
    )

# Add distributions for all three queries
add_distribution(fig, df_about, 'about_length', 1, 'skyblue')
add_distribution(fig, df_project, 'project_length', 2, 'lightgreen')
add_distribution(fig, df_combined, 'response_length', 3, 'coral')

# Update layout
fig.update_xaxes(title_text="Response Length (characters)", row=1, col=1)
fig.update_xaxes(title_text="Response Length (characters)", row=2, col=1)
fig.update_xaxes(title_text="Response Length (characters)", row=3, col=1)
fig.update_yaxes(title_text="Probability Density", row=1, col=1)
fig.update_yaxes(title_text="Probability Density", row=2, col=1)
fig.update_yaxes(title_text="Probability Density", row=3, col=1)

fig.update_layout(
    height=1200,
    title_text="Response Length Distribution Analysis",
    showlegend=True
)

fig.show()

In [ ]:
# Statistical Summary
summary_data = {
    'Question': ['About Yourself', 'Recent Project', 'Combined'],
    'Mean': [
        df_about['about_length'].mean(),
        df_project['project_length'].mean(),
        df_combined['response_length'].mean()
    ],
    'Median': [
        df_about['about_length'].median(),
        df_project['project_length'].median(),
        df_combined['response_length'].median()
    ],
    'Std Dev': [
        df_about['about_length'].std(),
        df_project['project_length'].std(),
        df_combined['response_length'].std()
    ],
    'Min': [
        df_about['about_length'].min(),
        df_project['project_length'].min(),
        df_combined['response_length'].min()
    ],
    'Max': [
        df_about['about_length'].max(),
        df_project['project_length'].max(),
        df_combined['response_length'].max()
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\nStatistical Summary:")
print(summary_df.to_string(index=False))

In [ ]:
# Box plot comparison
fig_box = go.Figure()

fig_box.add_trace(go.Box(
    y=df_about['about_length'],
    name='About Yourself',
    marker_color='skyblue'
))

fig_box.add_trace(go.Box(
    y=df_project['project_length'],
    name='Recent Project',
    marker_color='lightgreen'
))

fig_box.add_trace(go.Box(
    y=df_combined['response_length'],
    name='Combined',
    marker_color='coral'
))

fig_box.update_layout(
    title='Response Length Distribution: Box Plot Comparison',
    yaxis_title='Response Length (characters)',
    height=500
)

fig_box.show()